In [1]:
import pandas as pd
import numpy as np

def read_excel(file_name):
    df = pd.read_excel(file_name)
    return df

def read_txt(file_name):
    file = open(file_name)
    lines = file.readlines()
    return(lines[0])

In [2]:
import os
import glob

def get_files(subfolder, extension):
    dir = f"{os.getcwd()}/content/{subfolder}/"
    tables = glob.glob(f"{dir}*.{extension}")
    return tables

In [3]:
class Analizer:
    def __init__(self, boundary):
        self.results = get_files(subfolder="results", extension="xlsx")
        self.results_df = pd.DataFrame()
        self.boundary = boundary
    
    def has_minimum_requirements(self, df, sort_by="r2"):
        sorted_df = df.sort_values(by=sort_by, ascending=False)
        top_r2 = sorted_df.head(1)[sort_by].values[0]
        if top_r2 < self.boundary:  # aqui mantemos menor que o limite
            return True
        return False
    
    def concatenate_df(self, df, architecture):
        if self.has_minimum_requirements(df):
            df['Architecture'] = architecture
            df = df.rename(columns={'Unnamed: 0': 'model'})
            self.results_df = pd.concat([self.results_df, df], ignore_index=True) 

    def create_results_df(self):
        for file in self.results:
            df = read_excel(file)
            architecture = read_txt(file.replace(".xlsx", ".txt"))
            self.concatenate_df(df, architecture)
        self.results_df = self.results_df.sort_values(by="r2", ascending=False, ignore_index=True)

    def discard_below_average(self, sort_by):
        column_mean = self.results_df[sort_by].mean()      
        self.results_df = self.results_df[self.results_df[sort_by] >= column_mean]
    
    def discard_high_standard_deviation(self):
        r2_val, r2_test = self.results_df['r2_val'], self.results_df['r2_test']
        std_devs = np.abs(r2_val - r2_test)
        mean_std_dev = std_devs.mean()
        self.results_df = self.results_df[std_devs < mean_std_dev]

    def clean_folder(self, subfolder, extension, remove_last=True):
        files = get_files(subfolder, extension)
        models = self.results_df["model"]
        if (remove_last):
            models = models.apply(lambda x: '_'.join(x.rsplit('_', 1)[:-1]))
        for file in files:
            file_name = os.path.basename(file).split('.')[0]
            file_parts = file_name.split('_')            
            dataset_model = f"model_{file_parts[1]}_{file_parts[2]}" 
            if (remove_last == False):
                dataset_model = (f"{dataset_model}_{file_parts[3]}")
            if dataset_model not in models.values:
                os.remove(file)   
        
    def Analize(self):
        self.create_results_df()
        self.discard_below_average(sort_by="r2")
        self.discard_below_average(sort_by="r2_vt")
        self.discard_high_standard_deviation()
        self.results_df.to_excel(f"better_results.xlsx", index=True)
        display(self.results_df)


In [4]:
analize = Analizer(0.8)
analize.Analize()
analize.clean_folder(subfolder="dataset", extension="pkl")
analize.clean_folder(subfolder="results", extension="xlsx")
analize.clean_folder(subfolder="results", extension="txt")
analize.clean_folder(subfolder="models", extension="keras", remove_last=False)



,model,r2,r2_sup,r2_test,r2_val,r2_vt,mse,mse_sup,mse_test,mse_val,mse_vt,mape,rmse,r2_adj,rsd,aic,bic,Architecture
0,model_15_6_0,0.799928,0.149517,-0.171860,0.331319,0.275869,0.327620,1.392674,0.049663,0.226704,0.138183,0.987950,0.572381,1.009026,0.596748,1114.231803,1791.926762,"Hidden Size=[15, 30], regularizer=0.05, learni..."
1,model_3_9_0,0.799685,0.283440,0.760826,0.649020,0.820840,0.328017,1.173373,1.412208,0.054547,0.733378,0.557111,0.572728,1.018706,0.597110,564.229381,906.733487,"Hidden Size=[20, 10], regularizer=0.05, learni..."
2,model_12_5_10,0.799471,-0.019719,-14.477110,-0.925290,-2.800159,0.328369,1.669800,0.908193,0.573618,0.740906,2.554970,0.573035,1.011432,0.597430,892.227235,1434.626977,"Hidden Size=[16, 22], regularizer=0.05, learni..."
3,model_19_9_9,0.799436,0.070719,-1.263859,0.876827,0.619687,0.328425,1.521707,0.490730,0.180606,0.335668,0.870349,0.573083,1.010091,0.597481,1004.226896,1614.883684,"Hidden Size=[21, 19], regularizer=0.05, learni..."
4,model_9_5_12,0.799425,-0.029177,-7.727926,0.936692,0.677915,0.328444,1.685287,0.094944,0.020386,0.057665,0.785568,0.573100,1.011434,0.597498,892.226778,1434.626520,"Hidden Size=[16, 22], regularizer=0.2, learnin..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1947,model_1_5_4,0.358443,0.027887,0.369043,0.682263,0.465312,1.050556,1.591845,3.761026,0.438476,2.099750,1.064702,1.024966,1.059912,1.068601,561.901361,904.405467,"Hidden Size=[20, 10], regularizer=0.2, learnin..."
1948,model_17_2_16,0.357432,0.078516,0.304984,-5.660691,0.254545,1.052211,1.508939,1.029990,0.321453,0.675722,1.370231,1.025773,1.032330,1.069443,1001.898213,1612.555001,"Hidden Size=[21, 19], regularizer=0.2, learnin..."
1949,model_13_3_14,0.356988,0.089489,-0.807056,0.549420,0.419372,1.052938,1.490971,0.368237,0.739880,0.554059,0.824375,1.026128,1.029008,1.069812,1111.896831,1789.591790,"Hidden Size=[15, 30], regularizer=0.2, learnin..."
1951,model_1_4_14,0.355580,0.010812,-0.731601,0.288319,0.340099,1.055243,1.619804,0.511523,4.664221,2.587872,1.141881,1.027250,1.060179,1.070982,561.892458,904.396565,"Hidden Size=[20, 10], regularizer=0.2, learnin..."
